# Notebook 08 — Compound Schedules Analysis

Covers phenomena 14–20: CONC, ALT, CONJ, MULT, MIX, CHAIN, TAND.

Each section imports processed data from `data/processed/`, runs the
primary analyses described in the level specs, and saves figures to
`reports/figures/compound_schedules/`.

---
**Data requirements** (run `00_data_pipeline.ipynb` first):
- `data/processed/compound_schedules.parquet` — all compound-schedule events
- `references/level_specs/parameters.json` — IV parameters

**Sections**:
- A. Concurrent Schedules (CONC) — matching, COD effect
- B. Alternative Schedules (ALT) — dominant schedule analysis
- C. Conjunctive Schedules (CONJ) — IRI, binding constraint
- D. Multiple Schedules (MULT) — behavioral contrast, discrimination index
- E. Mixed Schedules (MIX) — MIX vs. MULT comparison
- F. Chained Schedules (CHAIN) — goal gradient, conditioned SR value
- G. Tandem Schedules (TAND) — TAND vs. CHAIN, IRT analysis


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy import stats
from scipy.optimize import minimize

DATA_DIR  = Path('../data/processed')
FIG_DIR   = Path('../reports/figures/compound_schedules')
SPECS_DIR = Path('../references/level_specs')
FIG_DIR.mkdir(parents=True, exist_ok=True)

with open(SPECS_DIR / 'parameters.json') as f:
    PARAMS = json.load(f)

# Load compound-schedule events
COMPOUND_PHENOMENA = [
    'concurrent_schedules', 'alternative_schedules', 'conjunctive_schedules',
    'multiple_schedules', 'mixed_schedules', 'chained_schedules', 'tandem_schedules'
]

df_all = pd.read_parquet(DATA_DIR / 'compound_schedules.parquet')
df_all['wall_time'] = pd.to_datetime(df_all['wall_time'])
print(f'Loaded {len(df_all):,} events across {df_all["participant_id"].nunique()} participants')
df_all.head()

---
## A. Concurrent Schedules (CONC)
### A1. Choice allocation and matching
GML: `log(B_L/B_R) = a·log(R_L/R_R) + log(b)`

In [ ]:
conc = df_all[df_all['phenomenon'] == 'concurrent_schedules'].copy()

# Compute allocation per participant per condition
def compute_conc_allocation(df):
    rows = []
    for (pid, cond), grp in df.groupby(['participant_id', 'condition']):
        B_L = (grp['event_code'] == 'L_RESP').sum()
        B_R = (grp['event_code'] == 'R_RESP').sum()
        R_L = (grp['event_code'] == 'L_SR').sum()
        R_R = (grp['event_code'] == 'R_SR').sum()
        switches = (grp['event_code'] == 'SWITCH').sum()
        if B_L + B_R > 0 and R_L + R_R > 0:
            rows.append({
                'participant_id': pid,
                'condition': cond,
                'B_L': B_L, 'B_R': B_R,
                'R_L': R_L, 'R_R': R_R,
                'p_left': B_L / (B_L + B_R),
                'r_left': R_L / (R_L + R_R),
                'log_B_ratio': np.log(B_L / max(B_R, 1)),
                'log_R_ratio': np.log(R_L / max(R_R, 1)),
                'switch_rate': switches
            })
    return pd.DataFrame(rows)

conc_alloc = compute_conc_allocation(conc)

# GML fit per condition
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
conditions = ['CONC_FR_FR', 'CONC_VI_VI', 'CONC_VI_VI_COD']
titles = ['CONC FR3 FR9\n(no COD)', 'CONC VI3s VI9s\n(no COD)', 'CONC VI3s VI9s\n(+ COD=2)']

for ax, cond, title in zip(axes, conditions, titles):
    sub = conc_alloc[conc_alloc['condition'] == cond]
    if len(sub) < 2:
        ax.set_title(f'{title}\n(insufficient data)')
        continue
    x = sub['log_R_ratio'].values
    y = sub['log_B_ratio'].values
    slope, intercept, r, p, se = stats.linregress(x, y)
    ax.scatter(x, y, alpha=0.7, s=60)
    xline = np.linspace(x.min()-0.5, x.max()+0.5, 100)
    ax.plot(xline, slope*xline + intercept, 'r-', linewidth=2,
            label=f'a={slope:.2f}, b={np.exp(intercept):.2f}\nr²={r**2:.2f}')
    ax.axline((0,0), slope=1, color='gray', linestyle='--', alpha=0.5, label='Strict matching')
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_xlabel('log(R_L / R_R)')
    ax.set_title(title)
    ax.legend(fontsize=9)

axes[0].set_ylabel('log(B_L / B_R)')
fig.suptitle('Concurrent Schedules — Generalized Matching Law', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'conc_matching.png', dpi=150)
plt.show()

In [ ]:
# A2. COD effect on switching rate
fig, ax = plt.subplots(figsize=(7, 5))
cod_compare = conc_alloc[conc_alloc['condition'].isin(['CONC_VI_VI', 'CONC_VI_VI_COD'])]
cod_compare.boxplot(column='switch_rate', by='condition', ax=ax)
ax.set_title('COD Effect on Switching Rate')
ax.set_xlabel('Condition')
ax.set_ylabel('Switches per session')
plt.suptitle('')
plt.tight_layout()
plt.savefig(FIG_DIR / 'conc_cod_switching.png', dpi=150)
plt.show()

if len(cod_compare) > 0:
    vi_sw = cod_compare[cod_compare['condition'] == 'CONC_VI_VI']['switch_rate']
    cod_sw = cod_compare[cod_compare['condition'] == 'CONC_VI_VI_COD']['switch_rate']
    if len(vi_sw) > 1 and len(cod_sw) > 1:
        t, p = stats.ttest_ind(vi_sw, cod_sw)
        print(f'COD switching t-test: t={t:.3f}, p={p:.3f}')

---
## B. Alternative Schedules (ALT)
### B1. Which component wins the race per condition

In [ ]:
alt = df_all[df_all['phenomenon'] == 'alternative_schedules'].copy()

# Count FIRST_FR vs FIRST_FI per condition
def alt_dominant(df):
    rows = []
    for (pid, cond), grp in df.groupby(['participant_id', 'condition']):
        first_fr = (grp['event_code'] == 'FIRST_FR').sum()
        first_fi = (grp['event_code'] == 'FIRST_FI').sum()
        total = first_fr + first_fi
        if total > 0:
            rows.append({'participant_id': pid, 'condition': cond,
                         'prop_FR': first_fr / total, 'n_trials': total})
    return pd.DataFrame(rows)

alt_dom = alt_dominant(alt)

fig, ax = plt.subplots(figsize=(8, 5))
cond_order = ['ALT_FR3_FI20s', 'ALT_FR5_FI10s', 'ALT_FR20_FI5s']
cond_labels = ['ALT FR3 FI20s\n(FR easy)', 'ALT FR5 FI10s\n(near tie)', 'ALT FR20 FI5s\n(FI easy)']
means = [alt_dom[alt_dom['condition']==c]['prop_FR'].mean() for c in cond_order]
sems  = [alt_dom[alt_dom['condition']==c]['prop_FR'].sem()  for c in cond_order]
bars = ax.bar(cond_labels, means, yerr=sems, capsize=5, color=['#4ade80','#facc15','#f87171'], alpha=0.85)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='50% (no dominance)')
ax.set_ylim(0, 1)
ax.set_ylabel('Proportion of trials where FR won first')
ax.set_title('Alternative Schedule — Dominant Component by Condition', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'alt_dominant_component.png', dpi=150)
plt.show()

---
## C. Conjunctive Schedules (CONJ)
### C1. Inter-reinforcement interval by condition

In [ ]:
conj = df_all[df_all['phenomenon'] == 'conjunctive_schedules'].copy()

def compute_iri(df):
    rows = []
    for (pid, cond), grp in df.groupby(['participant_id', 'condition']):
        sr_times = grp[grp['event_code'] == 'SR']['elapsed_s'].values
        if len(sr_times) > 1:
            iris = np.diff(sr_times)
            rows.append({'participant_id': pid, 'condition': cond,
                         'mean_IRI': iris.mean(), 'n_SRs': len(sr_times)})
    return pd.DataFrame(rows)

conj_iri = compute_iri(conj)

cond_order = ['pure_FR3', 'pure_FI8s', 'CONJ_FR3_FI3s', 'CONJ_FR3_FI8s', 'CONJ_FR8_FI3s']
fig, ax = plt.subplots(figsize=(10, 5))
iri_means = [conj_iri[conj_iri['condition']==c]['mean_IRI'].mean() for c in cond_order]
iri_sems  = [conj_iri[conj_iri['condition']==c]['mean_IRI'].sem()  for c in cond_order]
ax.bar(cond_order, iri_means, yerr=iri_sems, capsize=5,
       color=['#60a5fa','#34d399','#f59e0b','#f87171','#a78bfa'], alpha=0.85)
ax.set_ylabel('Mean IRI (s)')
ax.set_title('Conjunctive Schedules — Inter-Reinforcement Interval', fontweight='bold')
ax.set_xticklabels(cond_order, rotation=15)
plt.tight_layout()
plt.savefig(FIG_DIR / 'conj_iri.png', dpi=150)
plt.show()

In [ ]:
# C2. Binding constraint frequency
conj_wait = conj[conj['event_code'].isin(['WAIT_FI', 'WAIT_FR'])]
binding = conj_wait.groupby(['condition', 'event_code']).size().unstack(fill_value=0)
if not binding.empty:
    binding.plot(kind='bar', figsize=(9, 5), color=['#f87171', '#60a5fa'], alpha=0.85)
    plt.title('Binding Constraint Frequency (FR vs FI controls response)', fontweight='bold')
    plt.ylabel('Trial count')
    plt.xlabel('Condition')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'conj_binding_constraint.png', dpi=150)
    plt.show()

---
## D. Multiple Schedules (MULT)
### D1. Response rate by component and phase (behavioral contrast)

In [ ]:
mult = df_all[df_all['phenomenon'] == 'multiple_schedules'].copy()

def compute_mult_rates(df, resp_codes, seg_duration_s=20):
    rows = []
    for (pid, phase), grp in df.groupby(['participant_id', 'condition']):
        r1 = grp[grp['event_code'] == resp_codes[0]].shape[0]
        r2 = grp[grp['event_code'] == resp_codes[1]].shape[0]
        # Approximate time in each component
        segs = grp[grp['event_code'].isin(['SD1','SD2'])].shape[0]
        t_per_comp = segs * seg_duration_s / 2 if segs > 0 else seg_duration_s
        rows.append({
            'participant_id': pid, 'phase': phase,
            'rate_SD1': r1 / (t_per_comp / 60),
            'rate_SD2': r2 / (t_per_comp / 60),
            'disc_index': (r1 - r2) / (r1 + r2) if (r1 + r2) > 0 else np.nan
        })
    return pd.DataFrame(rows)

mult_rates = compute_mult_rates(mult, ['RESP_1','RESP_2'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
phases = ['A1', 'B', 'A2']
for ax, metric, title in zip(axes,
    [['rate_SD1','rate_SD2'], ['disc_index']],
    ['Response Rate by Component', 'Discrimination Index']):
    for m in metric:
        means = [mult_rates[mult_rates['phase']==p][m].mean() for p in phases]
        sems  = [mult_rates[mult_rates['phase']==p][m].sem()  for p in phases]
        ax.errorbar(phases, means, yerr=sems, marker='o', capsize=4, label=m)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Phase')
    ax.legend()

axes[0].set_ylabel('Responses per minute')
axes[1].set_ylabel('Discrimination index (R1-R2)/(R1+R2)')
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
fig.suptitle('Multiple Schedules — Behavioral Contrast (ABA Reversal)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'mult_contrast.png', dpi=150)
plt.show()

---
## E. Mixed Schedules (MIX)
### E1. MIX vs. MULT — EXT-component responding and discrimination

In [ ]:
mix = df_all[df_all['phenomenon'] == 'mixed_schedules'].copy()
# Combined with MULT for direct comparison
mix['schedule_type'] = 'MIX'
mult_comp = mult.copy()
mult_comp['schedule_type'] = 'MULT'
combined = pd.concat([mix, mult_comp], ignore_index=True)

# EXT-component response rate per schedule type
def ext_response_rate(df, sched_type, resp_code, seg_dur=20):
    rows = []
    for pid, grp in df[df['schedule_type']==sched_type].groupby('participant_id'):
        # Identify EXT segments: between COMP_CHANGE events where component=EXT
        # Use condition='A1' for MULT (EXT baseline)
        if sched_type == 'MIX':
            ext_grp = grp  # all data; EXT component mixed in
        else:
            ext_grp = grp[grp['condition'] == 'A1']
        # Proxy: count RESPONSE in EXT segments (observer codes COMP_CHANGE)
        # Use OMIT_2 for MULT, RESPONSE for MIX during EXT
        ext_resp = ext_grp[ext_grp['event_code'].isin(['OMIT_2','RESPONSE'])].shape[0]
        t_est = max(ext_grp['elapsed_s'].max() - ext_grp['elapsed_s'].min(), 1)
        rows.append({'participant_id': pid, 'schedule_type': sched_type,
                     'ext_rate_per_min': ext_resp / (t_est / 60)})
    return pd.DataFrame(rows)

mix_ext  = ext_response_rate(combined, 'MIX',  'RESPONSE')
mult_ext = ext_response_rate(combined, 'MULT', 'OMIT_2')
compare_df = pd.concat([mix_ext, mult_ext], ignore_index=True)

fig, ax = plt.subplots(figsize=(7, 5))
compare_df.boxplot(column='ext_rate_per_min', by='schedule_type', ax=ax)
ax.set_title('EXT Component Response Rate: MIX vs. MULT')
ax.set_ylabel('Responses / min (EXT segments)')
ax.set_xlabel('Schedule Type')
plt.suptitle('')
plt.tight_layout()
plt.savefig(FIG_DIR / 'mix_vs_mult_ext_rate.png', dpi=150)
plt.show()

if len(mix_ext) > 1 and len(mult_ext) > 1:
    t, p = stats.ttest_ind(mix_ext['ext_rate_per_min'], mult_ext['ext_rate_per_min'])
    print(f'MIX vs MULT EXT rate: t={t:.3f}, p={p:.3f}')
    print('Prediction: MIX > MULT (no SD to suppress EXT responding)')

In [ ]:
# E2. Within-session learning in MIX — does EXT rate decrease over alternations?
mix_only = mix.copy()
mix_only['segment_num'] = mix_only.groupby('participant_id').cumcount() // 10  # rough segment bins

if len(mix_only) > 0:
    seg_rates = (mix_only[mix_only['event_code']=='RESPONSE']
                 .groupby('segment_num').size() /
                 mix_only['participant_id'].nunique())
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(seg_rates.index, seg_rates.values, 'o-', color='#f87171')
    ax.set_xlabel('Segment number (proxy for time on task)')
    ax.set_ylabel('Mean responses per segment')
    ax.set_title('MIX — Within-Session Response Rate Across Alternations', fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'mix_within_session.png', dpi=150)
    plt.show()

---
## F. Chained Schedules (CHAIN)
### F1. Goal gradient — IRT by link position

In [ ]:
chain = df_all[df_all['phenomenon'] == 'chained_schedules'].copy()
chain = chain.sort_values(['participant_id','condition','elapsed_s'])

def compute_irt_by_link(df):
    rows = []
    for (pid, cond), grp in df.groupby(['participant_id','condition']):
        grp = grp.reset_index(drop=True)
        # Determine link from event codes
        for i, row in grp.iterrows():
            if row['event_code'] in ['L1_RESP','L2_RESP','L3_RESP']:
                link = int(row['event_code'][1])  # 1 or 2
                if i > 0:
                    prev_resp = grp[grp.index < i][grp['event_code'].str.endswith('RESP')]
                    if len(prev_resp) > 0:
                        irt = row['elapsed_s'] - prev_resp.iloc[-1]['elapsed_s']
                        rows.append({'participant_id': pid, 'condition': cond,
                                     'link': link, 'IRT': irt})
    return pd.DataFrame(rows)

chain_irt = compute_irt_by_link(chain)

if len(chain_irt) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    for cond in chain_irt['condition'].unique():
        sub = chain_irt[chain_irt['condition']==cond]
        means = sub.groupby('link')['IRT'].mean()
        ax.plot(means.index, means.values, 'o-', label=cond, linewidth=2)
    ax.set_xlabel('Link position (1=initial, 2=terminal)')
    ax.set_ylabel('Mean IRT (s)')
    ax.set_title('Chained Schedules — Goal Gradient (IRT by Link)', fontweight='bold')
    ax.legend()
    ax.invert_yaxis()  # Lower IRT = faster response; show gradient going down
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'chain_goal_gradient.png', dpi=150)
    plt.show()

In [ ]:
# F2. Chain completion rate and breaks
chain_trials = chain.groupby(['participant_id','condition']).apply(
    lambda g: pd.Series({
        'terminal_SRs': (g['event_code']=='TERM_SR').sum(),
        'chain_breaks': (g['event_code']=='CHAIN_BREAK').sum()
    })
).reset_index()

if len(chain_trials) > 0:
    chain_trials['completion_rate'] = (
        chain_trials['terminal_SRs'] /
        (chain_trials['terminal_SRs'] + chain_trials['chain_breaks']).clip(lower=1)
    )
    chain_trials.groupby('condition')['completion_rate'].mean().plot(
        kind='bar', figsize=(7,4), color='#4ade80', alpha=0.85
    )
    plt.title('Chain Completion Rate by Condition', fontweight='bold')
    plt.ylabel('Proportion of trials completed')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'chain_completion_rate.png', dpi=150)
    plt.show()

---
## G. Tandem Schedules (TAND)
### G1. TAND vs. CHAIN — Link 1 response rate and IRT pattern

In [ ]:
tand = df_all[df_all['phenomenon'] == 'tandem_schedules'].copy()

# Compute response rate in Component 1 for TAND and CHAIN FR3 FR3
def comp1_rate(df, resp_code):
    rows = []
    for pid, grp in df.groupby('participant_id'):
        c1 = grp[grp['event_code']==resp_code].shape[0]
        t = max(grp['elapsed_s'].max() - grp['elapsed_s'].min(), 1)
        rows.append({'participant_id': pid, 'c1_rate': c1 / (t/60)})
    return pd.DataFrame(rows)

tand_fr3fr3 = tand[tand['condition']=='TAND_FR3_FR3']
chain_fr3fr3 = chain[chain['condition']=='CHAIN_FR3_FR3']

tand_c1 = comp1_rate(tand_fr3fr3, 'RESPONSE')
chain_c1 = comp1_rate(chain_fr3fr3, 'L1_RESP')
tand_c1['type'] = 'TAND FR3 FR3'
chain_c1['type'] = 'CHAIN FR3 FR3'
tc_compare = pd.concat([tand_c1, chain_c1])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Response rate comparison
if len(tc_compare) > 0:
    tc_compare.boxplot(column='c1_rate', by='type', ax=axes[0])
    axes[0].set_title('Component 1 Response Rate\nTAND vs. CHAIN FR3 FR3')
    axes[0].set_ylabel('Responses / min')
    axes[0].set_xlabel('')
    plt.sca(axes[0])
    plt.suptitle('')

# Post-COMP1 pause comparison (conditioned SR effect)
def post_comp1_pause(df, comp1_code, sr_code):
    pauses = []
    for pid, grp in df.groupby('participant_id'):
        grp = grp.sort_values('elapsed_s').reset_index(drop=True)
        comp1_times = grp[grp['event_code']==comp1_code]['elapsed_s'].values
        next_resp_times = grp[grp['event_code'].str.endswith('RESP') | (grp['event_code']=='RESPONSE')]['elapsed_s'].values
        for t in comp1_times:
            after = next_resp_times[next_resp_times > t]
            if len(after) > 0:
                pauses.append(after[0] - t)
    return np.array(pauses)

tand_pauses  = post_comp1_pause(tand_fr3fr3,  'COMP1_DONE', 'TERM_SR')
chain_pauses = post_comp1_pause(chain_fr3fr3, 'L1_SR',      'TERM_SR')

if len(tand_pauses) > 0 and len(chain_pauses) > 0:
    axes[1].boxplot([tand_pauses, chain_pauses], labels=['TAND FR3 FR3', 'CHAIN FR3 FR3'])
    axes[1].set_title('Post-Component-1 Pause\n(TAND: no signal; CHAIN: blue coins appear)')
    axes[1].set_ylabel('Pause duration (s)')
    t, p = stats.ttest_ind(tand_pauses, chain_pauses)
    axes[1].set_xlabel(f't={t:.2f}, p={p:.3f}')

fig.suptitle('Tandem vs. Chained Schedules — Conditioned Reinforcer Effect', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'tand_vs_chain.png', dpi=150)
plt.show()

In [ ]:
# G2. IRT across component blocks — flat (TAND) vs. accelerating (CHAIN)
tand_irt = tand.copy()
tand_irt = tand_irt[tand_irt['event_code'] == 'RESPONSE'].sort_values(['participant_id','elapsed_s'])
tand_irt['IRT'] = tand_irt.groupby('participant_id')['elapsed_s'].diff()

chain_irt_c = chain[chain['event_code'].isin(['L1_RESP','L2_RESP'])].copy()
chain_irt_c['link'] = chain_irt_c['event_code'].str[1].astype(int)
chain_irt_c['IRT'] = chain_irt_c.groupby('participant_id')['elapsed_s'].diff()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, df_irt, title, color in zip(
    axes,
    [tand_irt, chain_irt_c],
    ['TAND — IRT Pattern (no gradient expected)', 'CHAIN — IRT by Link (goal gradient)'],
    ['#60a5fa', '#f87171']
):
    if 'link' not in df_irt.columns:
        df_irt = df_irt.copy()
        df_irt['hit_num'] = df_irt.groupby('participant_id').cumcount()
        means = df_irt.groupby('hit_num')['IRT'].mean().head(12)
        ax.plot(means.index, means.values, 'o-', color=color)
        ax.set_xlabel('Response number within trial')
    else:
        means = df_irt.groupby('link')['IRT'].mean()
        ax.bar(means.index, means.values, color=color, alpha=0.85)
        ax.set_xlabel('Link position')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Mean IRT (s)')

plt.tight_layout()
plt.savefig(FIG_DIR / 'tand_chain_irt_comparison.png', dpi=150)
plt.show()

---
## Summary Table — Compound Schedules Key Results

In [ ]:
summary = {
    'CONC': 'Matching: a≈1 (VI-VI), a<1 (FR-FR exclusive preference). COD reduces switching rate.',
    'ALT':  'FR dominates when ratio easy; FI dominates when interval short. Whichever is less demanding wins.',
    'CONJ': 'IRI > max(pure FR, pure FI). Binding constraint slows overall rate.',
    'MULT': 'Sharp discrimination (disc index near 1). Behavioral contrast in ABA phases.',
    'MIX':  'EXT-component responding higher than MULT (no SD to suppress). Low discrimination index.',
    'CHAIN':'Goal gradient: IRT decreases toward terminal link. Higher L1 rate than TAND.',
    'TAND': 'Flat IRT pattern. Longer post-comp1 pause than CHAIN (no conditioned SR at transition).'
}

print('Compound Schedules — Key Predicted Results\n' + '='*60)
for k, v in summary.items():
    print(f'\n{k}: {v}')